In [ ]:
import numpy as np

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Let's sample vectors from a cone around a given vector

We will try to take this in 3 steps.

1. Sample random vectors in a cone around the $z$-axis
2. Build a orthonormal basis to a given vector
3. Sample random vectors around a given vector

##  1. Sample vectors around $z$-axis

We first note that for a given $\psi$, we can apply a rotation about the $y$ axis to $\hat{z}$ to get a vector that is $\psi$ away from the $z$-axis, _i.e._

$$
\left(\begin{array}{ccc} 
\cos\psi & 0.0 & \sin\psi\\
 0.0 & 1.0 & 0.0 \\
 -\sin\psi & 0.0 & \cos\psi
\end{array}\right)
\left(\begin{array}{c} 0.0 \\ 0.0 \\ 1.0 \end{array}\right) = 
\left(\begin{array}{c} \sin\psi \\ 0.0 \\ \cos\psi \end{array}\right) =
\sin\psi \hat{x} + \cos\psi \hat{z}
$$

In [ ]:
psi = np.radians(30)

xhat = np.array([1, 0, 0])
yhat = np.array([0, 1, 0])
zhat = np.array([0, 0, 1])

v = np.sin(psi) * xhat + np.cos(psi) * zhat

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.quiver(0, 0, 0, v[0], v[1], v[2])

for basis in [xhat, yhat, zhat]:
    ax.quiver(0, 0, 0, basis[0], basis[1], basis[2], color="k")
    
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_zlim(0, 1)

plt.show()

That's a start, but now we would like to rotate this vector by a random amount about the $z$-axis, to give us that cone. Thus—for a random number, $\phi$, between $0$ and $2\pi$—we do:

$$
\left(\begin{array}{ccc} 
\cos\phi & -\sin\phi & 0.0\\
\sin\phi & \cos\phi & 0.0 \\
 0.0 & 0.0 & 1.0
\end{array}\right)
\left(\begin{array}{c} \sin\psi \\ 0.0 \\ \cos\psi \end{array}\right) = 
\left(\begin{array}{c} \cos\phi\sin\psi \\ \sin\phi\sin\psi \\ \cos\psi \end{array}\right) = 
\cos\phi\sin\psi \hat{x} + \sin\phi\sin\psi \hat{y} + \cos\psi \hat{z}
$$

In [ ]:
nsample = 100

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

for _ in range(nsample):
    phi = np.random.uniform(0, 2 * np.pi)
    v = np.cos(phi)*np.sin(psi)*xhat + np.sin(phi)*np.sin(psi)*yhat + np.cos(psi)*zhat
    ax.quiver(0, 0, 0, v[0], v[1], v[2], color="dodgerblue")

for basis in [xhat, yhat, zhat]:
    ax.quiver(0, 0, 0, basis[0], basis[1], basis[2], color="k")
    
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_zlim(0, 1)

plt.show()

Now we're sampling ! The key insight from this is that there was actually nothing special about the $\hat{x}$ and $\hat{y}$ vectors except that they formed an orthonormal basis with the $\hat{z}$ vector. Thus, if we are given a vector we can apply this same procedure as long as we can construct an orthonormal basis

## 2. Constructing an orthonormal basis

This is pretty straightforward linear algebra; however, there is one little numerical trick to watch out for

In [ ]:
from typing import Tuple

def orthonormal_basis(v: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    if len(v)!=3:
        raise ValueError("This only works for three vectors")
    v /= np.linalg.norm(v)
    # This is the tricky bit.
    # You can run into dumb numerical issues if you don't do this check
    if abs(v[0]) < 0.9:
        v1 = np.array([1.0, 0.0, 0.0])
    else:
        v1 = np.array([0.0, 1.0, 0.0])
    u = np.cross(v, v1)
    w = np.cross(v, u)
    u /= np.linalg.norm(u)
    w /= np.linalg.norm(w)
    return u, w

In [ ]:
np.random.seed(7)
v = np.random.uniform(-1, 1, 3)
u, w = orthonormal_basis(v)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.quiver(0, 0, 0, v[0], v[1], v[2], color="crimson")
ax.quiver(0, 0, 0, u[0], u[1], u[2], color="dodgerblue")
ax.quiver(0, 0, 0, w[0], w[1], w[2], color="dodgerblue")

for basis in [xhat, yhat, zhat]:
    ax.quiver(0, 0, 0, basis[0], basis[1], basis[2], color="k")

ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_zlim(-1, 1)

plt.show()

## 3. Sample random vectors around a vector

Now, we combine the first two steps to get that give a vector, $\vec{v}$, we can randomly sample from a cone around it by computing:

$$
\cos\phi\sin\psi \hat{u} + \sin\phi\sin\psi \hat{w} + \cos\psi \hat{v}
$$

where $\hat{v} = \vec{v} \,/\,\mathrm{norm}(v)$, $\phi$ is a random number between $0$ and $2\pi$, and $\hat{u}$ and $\hat{w}$ form an orthonormal basis with $\hat{v}$

In [ ]:
def sample_ring(v: np.ndarray, psi: float) -> np.ndarray:
#     v /= np.linalg.norm(v)
    u, w = orthonormal_basis(v)
    phi = np.random.uniform(0, 2 * np.pi)
    point = np.sin(psi) * (np.cos(phi) * u + np.sin(phi) * w) + np.cos(psi)*v
    return point

In [ ]:
np.random.seed(2)
nsample = 100
v = np.random.uniform(-1, 1, 3)

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.quiver(0, 0, 0, 2 * v[0], v[1], v[2], color="crimson")

for _ in range(nsample):
    u = sample_ring(v, psi)
    ax.quiver(0, 0, 0, u[0], u[1], u[2], color="dodgerblue")


for basis in [xhat, yhat, zhat]:
    ax.quiver(0, 0, 0, basis[0], basis[1], basis[2], color="k")

ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_zlim(-1, 1)

plt.show()

And just so you know I'm not lying to you about the angle between $\vec{v}$ and the generated vector actually being $\psi$

In [ ]:
v = np.random.uniform(-1, 1, 3)
u = sample_ring(v, psi)

print(f"The norm of u is {np.linalg.norm(u)}")
print(f"The angle between the vectors is {np.arccos(np.dot(v, u) / np.linalg.norm(v))}")
print(f"psi is {psi}")